# Data exploration

FNCE 2370 — Team 7 | Predicting Corporate Distress Risk

Inspect the supplied merged Parquet file, identifiers, outcome columns, historical coverage, and CHS feature availability.

Run from the repository root (or its `notebooks/` subdirectory). See `README.md` for execution instructions. Source construction conventions are documented following the TA clarification of November 24, 2026; see `README.md` for remaining verification limits.

## Exploration scope

We use the instructor's premerged CHS–Global Characteristics Parquet file; the TA confirmed that direct use is permitted when no further merge is needed. The supplied target is a narrower CRSP delisting-based failure measure (codes 550, 552, 560, 561, 574, 580, 584). Inspect metadata and selected columns to avoid loading the full 223-column panel into memory. `permno`, `permco`, dates, outcome flags and merge flags are diagnostics or identifiers, not candidate predictors. The full-period summaries are descriptive; subsequent feature screening and transformation fitting use only eligible development observations.

The supplementary `00_source_provenance_checks.ipynb` tests candidate upstream keys, event histories and the effect of choosing a single representative security per company-month. Those checks do not change the modeling sample.

In [1]:
import pandas as pd
from pathlib import Path

print(Path.cwd())

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

file_path = project_root / 'Project_5_data_chs_main_global.parquet'

print("File exists:", file_path.exists())

/Users/tanirya/Desktop/UPenn/Wharton 7th Semester/FNCE 2370/Final Project
File exists: True


In [2]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(file_path)

print("File size (GB):", round(file_path.stat().st_size / 1e9, 3))
print("Number of rows:", parquet_file.metadata.num_rows)
print("Number of columns:", parquet_file.metadata.num_columns)
print("Number of row groups:", parquet_file.metadata.num_row_groups)

print("\nColumn names:")
for col in parquet_file.schema_arrow.names:
    print(col)

File size (GB): 2.442
Number of rows: 3502382
Number of columns: 223
Number of row groups: 36

Column names:
permno
date
month_end
first_failure_month
first_exit_month
first_ambiguous_exit_month
at_risk
failure_current_month
failure_next_month
failure_t_plus_1
failure
nimta
nimtaavg
nita
tlta
tlmta
cashmta
mb
sigma
sigma_fill
rsize
price_log
exret
exretavg
portfolio_exret
ret_adj
sprtrn
failure_within_6m
failure_within_12m
failure_within_24m
failure_within_36m
global_factor_matched
global_factor_eom
global_factor_date
obs_main
exch_main
common
primary_sec
permco
gvkey
iid
id
excntry
size_grp
me
me_company
ret_exc_lead1m
ret_exc
ret
ret_local
ret_lag_dif
prc_local
prc_high
prc_low
bidask
curcd
fx
gics
naics
sic
ff49
dolvol
shares
tvol
adjfct
comp_tpci
crsp_shrcd
comp_exchg
crsp_exchcd
source_crsp
prc
market_equity
div12m_me
chcsho_12m
eqnpo_12m
ret_1_0
ret_3_1
ret_6_1
ret_9_1
ret_12_1
ret_12_7
ret_60_12
seas_1_1an
seas_1_1na
seas_2_5an
seas_2_5na
seas_6_10an
seas_6_10na
seas_11_15an
sea

In [3]:
import pandas as pd

batch = next(parquet_file.iter_batches(batch_size=10))
df_preview = batch.to_pandas()

pd.set_option("display.max_columns", None)

display(df_preview)

,permno,date,month_end,first_failure_month,first_exit_month,first_ambiguous_exit_month,at_risk,failure_current_month,failure_next_month,failure_t_plus_1,failure,nimta,nimtaavg,nita,tlta,tlmta,cashmta,mb,sigma,sigma_fill,rsize,price_log,exret,exretavg,portfolio_exret,ret_adj,sprtrn,failure_within_6m,failure_within_12m,failure_within_24m,failure_within_36m,global_factor_matched,global_factor_eom,global_factor_date,obs_main,exch_main,common,primary_sec,permco,gvkey,iid,id,excntry,size_grp,me,me_company,ret_exc_lead1m,ret_exc,ret,ret_local,ret_lag_dif,prc_local,prc_high,prc_low,bidask,curcd,fx,gics,naics,sic,ff49,dolvol,shares,tvol,adjfct,comp_tpci,crsp_shrcd,comp_exchg,crsp_exchcd,source_crsp,prc,market_equity,div12m_me,chcsho_12m,eqnpo_12m,ret_1_0,ret_3_1,ret_6_1,ret_9_1,ret_12_1,ret_12_7,ret_60_12,seas_1_1an,seas_1_1na,seas_2_5an,seas_2_5na,seas_6_10an,seas_6_10na,seas_11_15an,seas_11_15na,seas_16_20an,seas_16_20na,at_gr1,sale_gr1,capx_gr1,inv_gr1,debt_gr3,sale_gr3,capx_gr3,inv_gr1a,lti_gr1a,sti_gr1a,coa_gr1a,col_gr1a,cowc_gr1a,ncoa_gr1a,ncol_gr1a,nncoa_gr1a,fnl_gr1a,nfna_gr1a,tax_gr1a,be_gr1a,ebit_sale,gp_at,cop_at,ope_be,ni_be,ebit_bev,netis_at,eqnetis_at,dbnetis_at,oaccruals_at,oaccruals_ni,taccruals_at,taccruals_ni,noa_at,opex_at,at_turnover,sale_bev,rd_sale,cash_at,sale_emp_gr1,emp_gr1,ni_inc8q,noa_gr1a,ppeinv_gr1a,lnoa_gr1a,capx_gr2,saleq_gr1,niq_be,niq_at,niq_be_chg1,niq_at_chg1,rd5_at,dsale_dinv,dsale_drec,dgp_dsale,dsale_dsga,saleq_su,niq_su,capex_abn,op_atl1,gp_atl1,ope_bel1,cop_atl1,pi_nix,ocf_at,op_at,ocf_at_chg1,at_be,ocfq_saleq_std,tangibility,earnings_variability,aliq_at,f_score,o_score,z_score,kz_index,ni_ar1,ni_ivol,at_me,be_me,debt_me,netdebt_me,sale_me,ni_me,ocf_me,fcf_me,eqpo_me,eqnpo_me,rd_me,ival_me,bev_mev,ebitda_mev,aliq_mat,eq_dur,beta_60m,resff3_12_1,resff3_6_1,mispricing_mgmt,mispricing_perf,ivol_capm_21d,iskew_capm_21d,coskew_21d,beta_dimson_21d,ivol_ff3_21d,iskew_ff3_21d,ivol_hxz4_21d,iskew_hxz4_21d,rmax5_21d,rmax1_21d,rvol_21d,rskew_21d,zero_trades_21d,dolvol_126d,dolvol_var_126d,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_252d,ami_126d,ivol_capm_252d,prc_highprc_252d,betadown_252d,bidaskhl_21d,corr_1260d,betabab_1260d,rmax5_rvol_21d,age,qmj,qmj_prof,qmj_growth,qmj_safety
0,12941,1966-12-30,1966-12-31,None,1967-08-31,None,1,0,0,0,0,NaN,0.001435,NaN,NaN,NaN,NaN,NaN,0.348024,0.348024,-8.269509,2.708050,0.068464,-0.017870,0.070756,0.069264,-0.001492,0.0,NaN,NaN,NaN,1,1966-12-31,1966-12-30,1.0,1.0,1.0,1.0,6238.0,None,None,12941.0,USA,large,95.40375,95.40375,0.190405,0.065634,0.069264,0.069264,1.0,30.875,32.125,28.000,0.0,USD,1.0,None,0.0,3713.0,23.0,4.865900e+08,3.090,15760000.0,1.0,None,10.0,None,1.0,1.0,30.875,95.40375,0.000000,0.010597,-0.010541,0.069264,-0.067116,-0.225081,-0.347984,-0.368360,-0.135416,0.304955,-0.031250,-0.030396,-0.015272,0.010579,0.076664,0.009925,0.061868,0.020466,0.056404,-0.002200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.435617,-0.171080,-0.391885,NaN,0.096104,0.016114,0.204134,-0.049977,3.684026,0.015816,0.313511,NaN,NaN,0.028659,0.057269,0.020885,0.508599,0.001604,1.696200e+05,1.006640,0.001733,0.952812,0.001718,0.001785,0.125258,0.017996,0.598961,1.678613,0.009167,0.535310,1.552596,1.347521,492.0,NaN,NaN,NaN,-1.364950
1,12941,1967-01-31,1967-01-31,None,1967-08-31,None,1,0,0,0,0,NaN,0.001439,NaN,NaN,NaN,NaN,NaN,0.308590,0.308590,-8.166772,2.708050,0.102314,0.008835,0.116154,0.194332,0.078178,0.0,NaN,NaN,NaN,1,1967-01-31,1967-01-31,1.0,1.0,1.0,1.0,6238.0,None,None,12941.0,USA,large,113.94375,113.94375,0.026766,0.190405,0.194332,0.194332,1.0,36.875,38.000,31.625,0.0,USD,1.0,None,0.0,3713.0,23.0,1.097400e+09,3.090,29760000.0,1.0,

In [4]:
display(df_preview.dtypes)

permno                   int64
date                    object
month_end               object
first_failure_month     object
first_exit_month        object
                        ...   
age                    float64
qmj                    float64
qmj_prof               float64
qmj_growth             float64
qmj_safety             float64
Length: 223, dtype: object

In [5]:
# Firm and security identifiers
id_cols = [
    "permno",
    "permco",
    "gvkey",
    "iid",
    "id"
]

# Date variables
date_cols = [
    "date",
    "month_end",
    "first_failure_month",
    "first_exit_month",
    "first_ambiguous_exit_month",
    "global_factor_eom",
    "global_factor_date"
]

# Failure and risk-set indicators
target_cols = [
    "at_risk",
    "failure_current_month",
    "failure_next_month",
    "failure_t_plus_1",
    "failure",
    "failure_within_6m",
    "failure_within_12m",
    "failure_within_24m",
    "failure_within_36m"
]

# Sample and merge indicators
sample_cols = [
    "global_factor_matched",
    "obs_main",
    "exch_main",
    "common",
    "primary_sec",
    "excntry",
    "crsp_shrcd",
    "crsp_exchcd",
    "source_crsp"
]

# Baseline CHS predictors
chs_features = [
    "nimtaavg",
    "tlmta",
    "exretavg",
    "sigma",
    "rsize",
    "cashmta",
    "mb",
    "price_log"
]

# Additional variables for feature validation
auxiliary_cols = [
    "nimta",
    "nita",
    "tlta",
    "exret",
    "sigma_fill",
    "prc",
    "market_equity",
    "ret_adj",
    "sprtrn"
]

selected_cols = (
    id_cols
    + date_cols
    + target_cols
    + sample_cols
    + chs_features
    + auxiliary_cols
)

print("Selected columns:", len(selected_cols))

Selected columns: 47


In [6]:
df = pd.read_parquet(
    file_path,
    columns=selected_cols
)

print("Dataset shape:", df.shape)

print(
    "Memory usage (GB):",
    round(df.memory_usage(deep=True).sum() / 1e9, 3)
)

display(df.head())

Dataset shape: (3502382, 47)
Memory usage (GB): 2.449


,permno,permco,gvkey,iid,id,date,month_end,first_failure_month,first_exit_month,first_ambiguous_exit_month,global_factor_eom,global_factor_date,at_risk,failure_current_month,failure_next_month,failure_t_plus_1,failure,failure_within_6m,failure_within_12m,failure_within_24m,failure_within_36m,global_factor_matched,obs_main,exch_main,common,primary_sec,excntry,crsp_shrcd,crsp_exchcd,source_crsp,nimtaavg,tlmta,exretavg,sigma,rsize,cashmta,mb,price_log,nimta,nita,tlta,exret,sigma_fill,prc,market_equity,ret_adj,sprtrn
0,12941,6238.0,None,None,12941.0,1966-12-30,1966-12-31,None,1967-08-31,None,1966-12-31,1966-12-30,1,0,0,0.0,0.0,0.0,NaN,NaN,NaN,1,1.0,1.0,1.0,1.0,USA,10.0,1.0,1.0,0.001435,NaN,-0.017870,0.348024,-8.269509,NaN,NaN,2.70805,NaN,NaN,NaN,0.068464,0.348024,30.875,95.40375,0.069264,-0.001492
1,12941,6238.0,None,None,12941.0,1967-01-31,1967-01-31,None,1967-08-31,None,1967-01-31,1967-01-31,1,0,0,0.0,0.0,0.0,NaN,NaN,NaN,1,1.0,1.0,1.0,1.0,USA,10.0,1.0,1.0,0.001439,NaN,0.008835,0.308590,-8.166772,NaN,NaN,2.70805,NaN,NaN,NaN,0.102314,0.308590,36.875,113.94375,0.194332,0.078178
2,12941,6238.0,None,None,12941.0,1967-02-28,1967-02-28,None,1967-08-31,None,1967-02-28,1967-02-28,1,0,0,0.0,0.0,0.0,NaN,NaN,NaN,1,1.0,1.0,1.0,1.0,USA,10.0,1.0,1.0,0.001444,NaN,0.012261,0.327715,-8.139206,NaN,NaN,2.70805,NaN,NaN,NaN,0.028091,0.327715,38.000,117.42000,0.030508,0.001963
3,12941,6238.0,None,None,12941.0,1967-03-31,1967-03-31,None,1967-08-31,None,1967-03-31,1967-03-31,1,0,0,0.0,0.0,NaN,NaN,NaN,NaN,1,1.0,1.0,1.0,1.0,USA,10.0,1.0,1.0,0.001448,NaN,0.053015,0.487301,-7.944372,NaN,NaN,2.70805,NaN,NaN,NaN,0.194962,0.487301,48.000,148.32000,0.263158,0.039410
4,87995,6643.0,None,None,87995.0,1988-04-29,1988-04-30,None,1990-04-30,None,1988-04-30,1988-04-29,1,0,0,0.0,0.0,0.0,0.0,0.0,NaN,1,1.0,1.0,1.0,1.0,USA,10.0,3.0,1.0,-0.006248,NaN,0.029365,0.354403,-9.567433,NaN,NaN,2.70805,NaN,NaN,NaN,0.108402,0.354403,15.750,126.66150,0.125000,0.009425


In [7]:
for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("Date conversion completed.")
display(df[date_cols].dtypes)

Date conversion completed.


date                          datetime64[ns]
month_end                     datetime64[ns]
first_failure_month           datetime64[ns]
first_exit_month              datetime64[ns]
first_ambiguous_exit_month    datetime64[ns]
global_factor_eom             datetime64[ns]
global_factor_date            datetime64[ns]
dtype: object

In [8]:
print("First observation:", df["month_end"].min())
print("Last observation:", df["month_end"].max())

print("Unique securities:", df["permno"].nunique())
print("Unique CRSP companies:", df["permco"].nunique())
print("Unique Compustat companies:", df["gvkey"].nunique())

print("Unique months:", df["month_end"].nunique())

print(
    "Missing security identifiers:",
    df["permno"].isna().sum()
)

print(
    "Missing month-end dates:",
    df["month_end"].isna().sum()
)

First observation: 1963-01-31 00:00:00
Last observation: 2024-12-31 00:00:00
Unique securities: 26216
Unique CRSP companies: 25617
Unique Compustat companies: 23133
Unique months: 744
Missing security identifiers: 0
Missing month-end dates: 0


In [9]:
annual_coverage = (
    df.assign(year=df["month_end"].dt.year)
    .groupby("year")
    .agg(
        observations=("permno", "size"),
        unique_securities=("permno", "nunique"),
        unique_companies=("permco", "nunique"),
        current_month_failures=("failure_current_month", "sum"),
        global_matches=("global_factor_matched", "sum")
    )
)

pd.set_option("display.max_rows", 100)

display(annual_coverage)

,observations,unique_securities,unique_companies,current_month_failures,global_matches
year,,,,,
1963,23924,2087,2067,17,23864
1964,24450,2125,2109,6,24388
1965,24998,2175,2161,13,24924
1966,25305,2205,2188,11,25225
1967,25457,2240,2224,15,25356
1968,25505,2274,2251,9,25338
1969,26420,2335,2310,13,26179
1970,27652,2399,2367,11,27355
1971,28581,2495,2457,11,28203


## Identifier, target and feature diagnostics

Check uniqueness at both security-month and company-month levels before imposing sample restrictions. Compare the supplied failure indicators and their missingness instead of automatically shifting the target. Part I resolves their timing and risk-set interpretation; Part II uses explicit feature allowlists to exclude forward returns, future event dates and other outcome-derived fields.

In [10]:
print(
    "Duplicate security-month observations:",
    df.duplicated(
        subset=["permno", "month_end"]
    ).sum()
)

print(
    "Duplicate company-month observations:",
    df.loc[df["permco"].notna()].duplicated(
        subset=["permco", "month_end"]
    ).sum()
)

print(
    "Fully duplicated rows across loaded columns:",
    df.duplicated().sum()
)

Duplicate security-month observations: 0
Duplicate company-month observations: 35360
Fully duplicated rows across loaded columns: 0


In [11]:
for col in target_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- at_risk ---
at_risk
1    3480133
0      22249
Name: count, dtype: int64

--- failure_current_month ---
failure_current_month
0    3495976
1       6406
Name: count, dtype: int64

--- failure_next_month ---
failure_next_month
0    3495980
1       6402
Name: count, dtype: int64

--- failure_t_plus_1 ---
failure_t_plus_1
0.0    3468946
NaN      27034
1.0       6402
Name: count, dtype: int64

--- failure ---
failure
0.0    3468946
NaN      27034
1.0       6402
Name: count, dtype: int64

--- failure_within_6m ---
failure_within_6m
0.0    3338666
NaN     125702
1.0      38014
Name: count, dtype: int64

--- failure_within_12m ---
failure_within_12m
0.0    3185406
NaN     242150
1.0      74826
Name: count, dtype: int64

--- failure_within_24m ---
failure_within_24m
0.0    2896663
NaN     463822
1.0     141897
Name: count, dtype: int64

--- failure_within_36m ---
failure_within_36m
0.0    2637618
NaN     666165
1.0     198599
Name: count, dtype: int64


In [12]:
candidate_targets = [
    "failure_current_month",
    "failure_next_month",
    "failure_t_plus_1",
    "failure"
]

target_comparison = pd.DataFrame(
    index=candidate_targets,
    columns=candidate_targets,
    dtype="int64"
)

for col1 in candidate_targets:
    for col2 in candidate_targets:

        identical = (
            df[col1].eq(df[col2]).fillna(False)
            | (df[col1].isna() & df[col2].isna())
        )

        target_comparison.loc[col1, col2] = (
            (~identical).sum()
        )

display(target_comparison)

,failure_current_month,failure_next_month,failure_t_plus_1,failure
failure_current_month,0.0,12808.0,33436.0,33436.0
failure_next_month,12808.0,0.0,27034.0,27034.0
failure_t_plus_1,33436.0,27034.0,0.0,0.0
failure,33436.0,27034.0,0.0,0.0


In [13]:
for col in sample_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- global_factor_matched ---
global_factor_matched
1    3442786
0      59596
Name: count, dtype: int64

--- obs_main ---
obs_main
1.0    3442786
NaN      59596
Name: count, dtype: int64

--- exch_main ---
exch_main
1.0    3442786
NaN      59596
Name: count, dtype: int64

--- common ---
common
1.0    3442786
NaN      59596
Name: count, dtype: int64

--- primary_sec ---
primary_sec
1.0    3442786
NaN      59596
Name: count, dtype: int64

--- excntry ---
excntry
USA     3442786
None      59596
Name: count, dtype: int64

--- crsp_shrcd ---
crsp_shrcd
10.0    3442786
NaN       59596
Name: count, dtype: int64

--- crsp_exchcd ---
crsp_exchcd
3.0    1954946
1.0    1043354
2.0     444486
NaN      59596
Name: count, dtype: int64

--- source_crsp ---
source_crsp
1.0    3442786
NaN      59596
Name: count, dtype: int64


In [14]:
feature_summary = pd.DataFrame({
    "non_missing": df[chs_features].notna().sum(),
    "missing": df[chs_features].isna().sum(),
    "missing_pct": df[chs_features].isna().mean() * 100,
    "mean": df[chs_features].mean(),
    "std": df[chs_features].std(),
    "min": df[chs_features].min(),
    "max": df[chs_features].max()
})

display(feature_summary.round(4))

,non_missing,missing,missing_pct,mean,std,min,max
nimtaavg,3422715,79667,2.2747,-0.0049,0.0499,-6.3411,3.7645
tlmta,2642574,859808,24.5492,0.4328,0.2849,-1.1570,0.9999
exretavg,3480467,21915,0.6257,-0.0088,0.0586,-1.8905,0.7661
sigma,3375151,127231,3.6327,0.5611,0.4832,0.0076,80.1287
rsize,3405962,96420,2.7530,-10.4229,2.0701,-19.9076,-2.2432
cashmta,2629503,872879,24.9224,0.1093,0.1792,-0.1746,10.9551
mb,2740008,762374,21.7673,17.9954,683.8753,0.0015,148664.7808
price_log,3406073,96309,2.7498,2.0124,0.9930,-4.5564,2.7081
